# 05B Global Model Transfer Only

This notebook runs a strict saved-artifact transfer evaluation on M5 using synthetic-trained global boosting artifacts.

Important:
- this is an honest saved-model transfer test
- it does **not** retrain on M5
- it produces transfer metrics on monthly aggregated M5 data
- it is not the same as a Kaggle daily submission, because the saved synthetic artifacts are monthly models. This notebook is transfer evaluation only and does not generate uploadable XGBOOST/CATBOOST Kaggle CSVs.
- prerequisite: saved artifacts must already exist under `modeling/outputs/artifacts` (for example from notebook 02 global model training and artifact save)

In [1]:
from pathlib import Path
import pandas as pd

M5_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy')
REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')
SCRIPT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py')
TAG = 'portable_m5_transfer_auto'

def resolve_report_path(kind: str, preferred_tag: str = TAG) -> Path:
    if not REPORTS_DIR.exists():
        raise FileNotFoundError(
            f'Reports directory not found: {REPORTS_DIR}. Run the transfer cell above first.'
        )

    preferred = REPORTS_DIR / f'{preferred_tag}_{kind}.csv'
    if preferred.exists():
        return preferred

    available = sorted(p.name for p in REPORTS_DIR.glob(f'{preferred_tag}_*.csv'))
    raise FileNotFoundError(
        f'Missing transfer report: {preferred.name}. Run the transfer cell above first. '
        f'Available transfer CSVs for tag {preferred_tag!r}: {available}'
    )


## Run Transfer Evaluation

In [2]:
ARTIFACTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts')
required_checks = [
    ARTIFACTS_DIR / 'P' / 'xgboost_h1' / 'production' / 'metadata.json',
]
missing = [str(p) for p in required_checks if not p.exists()]

if missing:
    raise FileNotFoundError(
        'Missing saved model artifacts required for transfer evaluation. '
        'Run notebook 02_global_model_training_and_artifact_save.ipynb first. '
        f'Missing examples: {missing}'
    )

!python "{SCRIPT}" --m5-dir "{M5_DIR}" --granularity dept_store --datasets P --models XGBOOST --tag "{TAG}" --calibration recent_level_auto


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_summary.csv
Saved calibration: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_calibration.csv


## Summary

In [3]:
!python "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py" \
  --m5-dir "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy" \
  --granularity dept_store \
  --datasets P \
  --models XGBOOST \
  --tag "portable_m5_transfer_auto" \
  --calibration recent_level_auto


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_summary.csv
Saved calibration: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/portable_m5_transfer_auto_calibration.csv


In [4]:
summary_path = resolve_report_path('summary')
pd.read_csv(summary_path)


,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,0,10080,0.129355,3832.483612,-927.898585,1.153012,1095.152089,0.654762


## Detailed Horizon Metrics

In [5]:
metrics_path = resolve_report_path('metrics')
pd.read_csv(metrics_path).head(50)

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,1,840,0.091335,2835.751123,-270.023571,0.781425,773.267897,0.547619
1,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,2,840,0.103036,3206.873134,-457.663492,0.921605,872.330556,0.569048
2,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,3,840,0.110401,3401.122417,-597.745794,1.001863,934.685635,0.580952
3,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,4,840,0.113020,3477.637540,-725.803135,1.023718,956.859306,0.596429
4,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,5,840,0.126420,3746.175711,-820.036627,1.129481,1070.302044,0.625000
5,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,6,840,0.129690,3789.908125,-964.109881,1.147416,1097.984901,0.633333
6,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,7,840,0.138539,4065.233471,-1067.546429,1.248431,1172.904286,0.652381
7,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,8,840,0.139222,4127.712864,-1142.285714,1.264716,1178.685873,0.698810
8,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,9,840,0.142403,4168.131562,-1219.925278,1.264759,1205.618591,0.710714
9,M5_TRANSFER_P,XGBOOST_recent_level_auto,test,10,840,0.147749,4196.182511,-1263.825635,1.317678,1250.879762,0.733333


## Why This Is Not a Kaggle Submission

The saved artifacts are monthly synthetic-trained models. Kaggle M5 submission requires 28-day daily item-store forecasts. That means:
- this notebook is valid for transfer evaluation
- it is not valid for strict Kaggle submission generation from the same saved monthly artifacts